In [3]:
import cv2
import numpy as np
import tensorflow as tf
import subprocess
import threading
import time
import os
from queue import Queue

def process_video():
    """Process video with YOLOv8 and spatial detection feedback."""
    print("Starting video processing...")
    
    # Load YOLOv8 model
    try:
        from ultralytics import YOLO
        model = YOLO("yolov8n")
        print("YOLOv8 model loaded successfully")
    except ImportError:
        print("Error: Ultralytics package not found.")
        print("To install: pip install ultralytics")
        return
    except Exception as e:
        print(f"Error loading model: {e}")
        return
    
    # Video input/output using fixed filenames
    cap = cv2.VideoCapture("amstest.mp4")
    if not cap.isOpened():
        print("Error: Could not open video file")
        return
        
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter("feedback test1.mp4", fourcc, fps, (width, height))
    
    # Calculate screen sections
    left_boundary = width // 3
    right_boundary = (width * 2) // 3
    
    # Custom class categories
    classes = [
        'crosswalk',
        'obstacle',
        'pedestrians',
        'signal',
        'stairs',
        'vehicle'
    ]
    
    # Detection parameters
    confidence_threshold = 0.5
    
    # Feedback display settings
    current_feedback = ""
    feedback_display_time = 3  # seconds to display text
    feedback_start_time = 0
    last_announcement = {}  # Store last announcement time for each class-position pair
    announcement_cooldown = 3  # Seconds between repeated announcements
        
    def determine_position(self, x_center):
        """Determine which screen section a detection falls into based on its center x-coordinate."""
        if x_center < self.left_boundary:
            return "left"
        elif x_center > self.right_boundary:
            return "right"
        else:
            return "center"
    
    def add_feedback(self, obj_class, position):
        """Add feedback to the queue if it hasn't been announced recently."""
        key = f"{obj_class}-{position}"
        current_time = time.time()
        
        # Check if this announcement was made recently
        if key not in self.last_announcement or \
           (current_time - self.last_announcement[key]) > self.announcement_cooldown:
            
            # Update the last announcement time
            self.last_announcement[key] = current_time
            
            # Add to feedback queue
            feedback_text = f"{obj_class} on the {position}"
            self.current_feedback = feedback_text
            self.feedback_start_time = current_time
    
    def process_detections(self, results, frame, frame_time):
        """Process YOLOv8 detection results."""
        # Create a visualization frame
        vis_frame = frame.copy()
        
        # Draw section boundaries
        cv2.line(vis_frame, (self.left_boundary, 0), (self.left_boundary, self.frame_height), (0, 255, 0), 2)
        cv2.line(vis_frame, (self.right_boundary, 0), (self.right_boundary, self.frame_height), (0, 255, 0), 2)
        
        # Add section labels
        cv2.putText(vis_frame, "LEFT", (self.left_boundary // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(vis_frame, "CENTER", ((self.left_boundary + self.right_boundary) // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(vis_frame, "RIGHT", (self.right_boundary + (self.frame_width - self.right_boundary) // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        
        # Count pedestrians for special handling
        pedestrian_count = 0
        pedestrian_positions = []
        
        # For the Ultralytics YOLOv8 implementation:
        try:
            for result in results:
                boxes = result.boxes
                
                # First pass: count pedestrians
                for box in boxes:
                    # Get class ID
                    class_id = int(box.cls)
                    class_name = self.classes[class_id] if class_id < len(self.classes) else f"Class {class_id}"
                    
                    # Count pedestrians
                    if class_name == 'pedestrians':
                        pedestrian_count += 1
                        # Store position for later use
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        x_center = (x1 + x2) / 2
                        pedestrian_positions.append(self.determine_position(x_center))
                
                # Handle special case for crowd (5+ pedestrians)
                if pedestrian_count >= 5:
                    self.current_feedback = "crowd ahead"
                    self.feedback_start_time = frame_time
                
                # Second pass: process all detections
                for box in boxes:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    
                    # Get class ID and confidence
                    class_id = int(box.cls)
                    confidence = float(box.conf)
                    
                    # Get class name
                    class_name = self.classes[class_id] if class_id < len(self.classes) else f"Class {class_id}"
                    
                    # Calculate center point
                    x_center = (x1 + x2) / 2
                    
                    # Determine position
                    position = self.determine_position(x_center)
                    
                    # Apply the special feedback rules
                    if class_name == 'stairs':
                        # Always provide feedback for stairs
                        self.add_feedback(class_name, position)
                    elif class_name == 'pedestrians':
                        # Only provide feedback for a single pedestrian (not 2-4)
                        # For 5+ pedestrians, "crowd ahead" was already added above
                        if pedestrian_count == 1:
                            self.add_feedback(class_name, position)
                        # Don't provide feedback for 2-4 pedestrians
                    else:
                        # Normal feedback for all other classes
                        self.add_feedback(class_name, position)
                    
                    # Draw bounding box
                    color = (255, 0, 0) if position == "left" else \
                            (0, 0, 255) if position == "right" else (0, 255, 255)
                    
                    cv2.rectangle(vis_frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                    
                    # Add label with count for pedestrians when there are multiple
                    if class_name == 'pedestrians' and pedestrian_count > 1:
                        text = f"{class_name} ({pedestrian_count}) ({position})"
                    else:
                        text = f"{class_name} ({position})"
                    
                    cv2.putText(vis_frame, text, (int(x1), int(y1) - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        except Exception as e:
            print(f"Error processing YOLOv8 results: {e}")
            
        # Add feedback text at the bottom middle of the screen
        # Check if we should still display the current feedback
        if frame_time - self.feedback_start_time < self.feedback_display_time:
            # Add a semi-transparent black background for better text visibility
            text_size = cv2.getTextSize(self.current_feedback, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
            text_x = (self.frame_width - text_size[0]) // 2
            text_y = self.frame_height - 50
            
            # Draw background rectangle
            cv2.rectangle(vis_frame, 
                         (text_x - 10, text_y - text_size[1] - 10), 
                         (text_x + text_size[0] + 10, text_y + 10), 
                         (0, 0, 0, 0.7), 
                         -1)
            
            # Draw text
            cv2.putText(vis_frame, 
                       self.current_feedback, 
                       (text_x, text_y), 
                       cv2.FONT_HERSHEY_SIMPLEX, 
                       1.2, 
                       (255, 255, 255), 
                       3)
        
        return vis_frame
    
    # Helper functions
    def determine_position(x_center, left_boundary, right_boundary):
        """Determine which screen section a detection falls into based on its center x-coordinate."""
        if x_center < left_boundary:
            return "left"
        elif x_center > right_boundary:
            return "right"
        else:
            return "center"
    
    def add_feedback(obj_class, position, last_announcement, current_time):
        """Add feedback if it hasn't been announced recently."""
        key = f"{obj_class}-{position}"
        
        # Check if this announcement was made recently
        if key not in last_announcement or \
           (current_time - last_announcement[key]) > announcement_cooldown:
            
            # Update the last announcement time
            last_announcement[key] = current_time
            
            # Return feedback text
            return f"{obj_class} on the {position}"
        
        return None
    
    # Variables for progress tracking
    frame_count = 0
    start_time = time.time()
    last_update_time = start_time
    
    # Process frames
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        # Current frame time (used for feedback display timing)
        frame_time = time.time()
        
        # Create a visualization frame with section boundaries
        vis_frame = frame.copy()
        cv2.line(vis_frame, (left_boundary, 0), (left_boundary, height), (0, 255, 0), 2)
        cv2.line(vis_frame, (right_boundary, 0), (right_boundary, height), (0, 255, 0), 2)
        
        # Add section labels
        cv2.putText(vis_frame, "LEFT", (left_boundary // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(vis_frame, "CENTER", ((left_boundary + right_boundary) // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(vis_frame, "RIGHT", (right_boundary + (width - right_boundary) // 2, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        
        # Run YOLOv8 inference
        results = model.predict(frame, conf=confidence_threshold)
        
        # Count pedestrians for special handling
        pedestrian_count = 0
        pedestrian_positions = []
        
        # First pass: count pedestrians
        try:
            for result in results:
                boxes = result.boxes
                
                for box in boxes:
                    # Get class ID
                    class_id = int(box.cls)
                    class_name = classes[class_id] if class_id < len(classes) else f"Class {class_id}"
                    
                    # Count pedestrians
                    if class_name == 'pedestrians':
                        pedestrian_count += 1
                        # Store position for later use
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        x_center = (x1 + x2) / 2
                        pedestrian_positions.append(determine_position(x_center, left_boundary, right_boundary))
                
                # Handle special case for crowd (5+ pedestrians)
                if pedestrian_count >= 5:
                    current_feedback = "crowd ahead"
                    feedback_start_time = frame_time
                
                # Second pass: process all detections
                for box in boxes:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    
                    # Get class ID and confidence
                    class_id = int(box.cls)
                    confidence = float(box.conf)
                    
                    # Get class name
                    class_name = classes[class_id] if class_id < len(classes) else f"Class {class_id}"
                    
                    # Calculate center point
                    x_center = (x1 + x2) / 2
                    
                    # Determine position
                    position = determine_position(x_center, left_boundary, right_boundary)
                    
                    # Apply the special feedback rules
                    if class_name == 'stairs':
                        # Always provide feedback for stairs
                        feedback = add_feedback(class_name, position, last_announcement, frame_time)
                        if feedback:
                            current_feedback = feedback
                            feedback_start_time = frame_time
                    elif class_name == 'pedestrians':
                        # Only provide feedback for a single pedestrian (not 2-4)
                        # For 5+ pedestrians, "crowd ahead" was already added above
                        if pedestrian_count == 1:
                            feedback = add_feedback(class_name, position, last_announcement, frame_time)
                            if feedback:
                                current_feedback = feedback
                                feedback_start_time = frame_time
                        # Don't provide feedback for 2-4 pedestrians
                    else:
                        # Normal feedback for all other classes
                        feedback = add_feedback(class_name, position, last_announcement, frame_time)
                        if feedback:
                            current_feedback = feedback
                            feedback_start_time = frame_time
                    
                    # Draw bounding box
                    color = (255, 0, 0) if position == "left" else \
                            (0, 0, 255) if position == "right" else (0, 255, 255)
                    
                    cv2.rectangle(vis_frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                    
                    # Add label with count for pedestrians when there are multiple
                    if class_name == 'pedestrians' and pedestrian_count > 1:
                        text = f"{class_name} ({pedestrian_count}) ({position})"
                    else:
                        text = f"{class_name} ({position})"
                    
                    cv2.putText(vis_frame, text, (int(x1), int(y1) - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        except Exception as e:
            print(f"Error processing YOLOv8 results: {e}")
        
        # Add feedback text at the bottom middle of the screen
        # Check if we should still display the current feedback
        if frame_time - feedback_start_time < feedback_display_time and current_feedback:
            # Add a semi-transparent black background for better text visibility
            text_size = cv2.getTextSize(current_feedback, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
            text_x = (width - text_size[0]) // 2
            text_y = height - 50
            
            # Draw background rectangle
            cv2.rectangle(vis_frame, 
                         (text_x - 10, text_y - text_size[1] - 10), 
                         (text_x + text_size[0] + 10, text_y + 10), 
                         (0, 0, 0), 
                         -1)
            
            # Draw text
            cv2.putText(vis_frame, 
                       current_feedback, 
                       (text_x, text_y), 
                       cv2.FONT_HERSHEY_SIMPLEX, 
                       1.2, 
                       (255, 255, 255), 
                       3)
        
        # Write frame to output video
        out.write(vis_frame)
        
        # Update progress
        frame_count += 1
        current_time = time.time()
        if current_time - last_update_time >= 5:  # Update every 5 seconds
            elapsed_time = current_time - start_time
            fps = frame_count / elapsed_time
            percent_complete = (frame_count / total_frames) * 100
            eta = (total_frames - frame_count) / fps if fps > 0 else 0
            
            print(f"Progress: {percent_complete:.1f}% ({frame_count}/{total_frames}) | "
                  f"FPS: {fps:.2f} | ETA: {eta:.1f} seconds")
            
            last_update_time = current_time
    
    # Clean up
    cap.release()
    out.release()
    
    # Print completion information
    total_time = time.time() - start_time
    print(f"Video processing complete!")
    print(f"Processed {frame_count} frames in {total_time:.2f} seconds ({frame_count/total_time:.2f} fps)")
    print(f"Output saved to: output_optimized.mp4")


if __name__ == "__main__":
    process_video()

Starting video processing...
YOLOv8 model loaded successfully

0: 256x416 2 vehicles, 13.3ms
Speed: 2.4ms preprocess, 13.3ms inference, 2.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 12.3ms
Speed: 2.1ms preprocess, 12.3ms inference, 4.3ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 12.9ms
Speed: 2.2ms preprocess, 12.9ms inference, 4.5ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 18.8ms
Speed: 2.2ms preprocess, 18.8ms inference, 5.8ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 25.8ms
Speed: 2.7ms preprocess, 25.8ms inference, 2.8ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 12.9ms
Speed: 2.1ms preprocess, 12.9ms inference, 5.4ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 17.4ms
Speed: 1.7ms preprocess, 17.4ms inference, 8.1ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 vehicles, 48.9ms
Speed: 4.8m